# AI-Powered Plant Identification with ArcGIS

This notebook uses the **ArcGIS AI Utility Services** (`analyze_image`) to automatically identify plants from photos attached to a hosted Feature Layer. For each unidentified feature, it fetches the photo attachment, sends it to the AI service with a set of targeted prompts, and writes the results — common name, latin name, plant type, and estimated height — back to the layer.

---

## Step 0 – Prerequisites

Import the required libraries. `analyze_image` from `arcgis.ai` is the core AI service used throughout this notebook.

In [1]:
# Step 0 – prerequisites
from pathlib import Path
import tempfile
from arcgis.gis import GIS
from arcgis.ai import analyze_image

## Step 1 – Connect to ArcGIS Online

Authenticate with ArcGIS Online using a username and password. The password is entered securely via a prompt so credentials are never stored in the notebook.

In [2]:
import getpass

# Connect to ArcGIS Online – password entered securely via prompt
username = input("ArcGIS Online username: ")
password = getpass.getpass("ArcGIS Online password: ")
gis = GIS("https://www.arcgis.com", username, password)


## Step 2 – Load the Feature Layer

Retrieve the target Feature Service item by its item ID and access the first layer, which contains the plant survey records with photo attachments.

In [4]:
# Step 1 – get the Feature Layer (layer 0 of the item)
item_id = "cad81fb939764686a3b1db0151fab2ff"
fl_item = gis.content.get(item_id)
feature_layer = fl_item.layers[0]


## Step 3 – Query Unidentified Features

Query only features where the common name field is `NULL` — these are the records that still need AI analysis. This avoids re-processing features that have already been identified.

In [5]:
# Step 2 – query features where the common name field is NULL
field_common = "common_name_enter_the_common_na"
where_clause = f"{field_common} IS NULL"
features = feature_layer.query(where=where_clause).features
print(f"{len(features)} features to process")


2 features to process


## Step 4 – Define AI Prompts

Define the list of prompts to send to the AI image analysis service. Each entry maps a target field name (`key`) to an instruction (`context`) that tells the model exactly what to return. All four prompts are sent in a single API call per feature.

In [6]:
# Step 3 – define prompts for the four desired attributes
prompt_data = [
    {
        "key": "common_name_enter_the_common_na",
        "context": (
            "Locate the main plant in the image then try to identify its common English name. "
            "Return only the common name."
        ),
    },
    {
        "key": "latin_name_enter_the_latin_name",
        "context": (
            "Locate the main plant in the image then try to identify its latin name. "
            "Return only the latin name."
        ),
    },
    {
        "key": "type_select_whether_the_plant_i",
        "context": (
            "Locate the main plant in the image then try to identify its type. "
            "Return only one word: Tree or Shrub."
        ),
    },
    {
        "key": "height_meters_enter_the_height",
        "context": (
            "Locate the main plant in the image then estimate its height. "
            "You should only return a number in meters formatted as 'n.n'."
        ),
    },
]


## Step 5 – Analyse Attachments and Collect Updates

For each queried feature:
1. Retrieve the list of photo attachments for that feature.
2. Build a tokenised HTTPS URL pointing to the first attachment.
3. Send all prompts to `analyze_image` in a single call.
4. Extract the plain-text results keyed by field name and stage them as attribute updates.

In [7]:
# Step 4 – iterate over features, analyse attachment, collect updates
updates = []        # list of feature dicts to submit via edit_features
results_dict = {}   # {OBJECTID: {field: value, ...}}

oid_field = feature_layer.properties.objectIdField   # e.g. "OBJECTID"
print(f"Processing {len(features)} features...")

for i, feat in enumerate(features, 1):
    oid = feat.attributes[oid_field]
    print(f"[{i}/{len(features)}] OID {oid} – fetching attachment...")

    atts_info = feature_layer.attachments.get_list(oid)
    if not atts_info:
        print(f"  Skipping OID {oid}: no attachment found.")
        continue  # skip features with no photo attachment

    att_id = atts_info[0]["id"]
    img_url = f"{feature_layer.url}/{oid}/attachments/{att_id}?token={gis._con.token}"

    try:
        results = analyze_image(img_url, prompt_data).results
    except Exception as exc:
        print(f"  OID {oid}: analyze_image failed – {exc}")
        continue

    # extract plain string values keyed by field name
    new_attrs = {r.key: (r.value.strip() if isinstance(r.value, str) else r.value)
                 for r in results}

    if not new_attrs:
        print(f"  Skipping OID {oid}: no results returned.")
        continue

    results_dict[oid] = new_attrs
    updates.append({"attributes": {oid_field: oid, **new_attrs}})
    print(f"  OID {oid}: {new_attrs}")

print(f"\nDone. {len(updates)} feature(s) ready to update.")


Processing 2 features...
[1/2] OID 7 – fetching attachment...
  OID 7: {'common_name_enter_the_common_na': 'Calla Lily', 'latin_name_enter_the_latin_name': 'Zantedeschia', 'type_select_whether_the_plant_i': 'Shrub', 'height_meters_enter_the_height': 0.4}
[2/2] OID 8 – fetching attachment...
  OID 8: {'common_name_enter_the_common_na': 'Rubber plant', 'latin_name_enter_the_latin_name': 'Ficus elastica', 'type_select_whether_the_plant_i': 'Tree', 'height_meters_enter_the_height': 1.6}

Done. 2 feature(s) ready to update.


In [8]:
updates

[{'attributes': {'objectid': 7,
   'common_name_enter_the_common_na': 'Calla Lily',
   'latin_name_enter_the_latin_name': 'Zantedeschia',
   'type_select_whether_the_plant_i': 'Shrub',
   'height_meters_enter_the_height': 0.4}},
 {'attributes': {'objectid': 8,
   'common_name_enter_the_common_na': 'Rubber plant',
   'latin_name_enter_the_latin_name': 'Ficus elastica',
   'type_select_whether_the_plant_i': 'Tree',
   'height_meters_enter_the_height': 1.6}}]

## Step 6 – Write Results Back to the Feature Layer

Apply all staged attribute updates to the Feature Layer in a single `edit_features` call. The final `results_dict` is displayed below for a quick in-notebook review of what was written.

In [9]:
# Step 5 – apply attribute updates back to the feature layer
if updates:
    edit_result = feature_layer.edit_features(updates=updates)
    print("Update status:", edit_result)
else:
    print("No updates prepared – nothing to edit on the layer.")

results_dict


Update status: {'addResults': [], 'updateResults': [{'objectId': 7, 'uniqueId': 7, 'globalId': None, 'success': True}, {'objectId': 8, 'uniqueId': 8, 'globalId': None, 'success': True}], 'deleteResults': []}


{7: {'common_name_enter_the_common_na': 'Calla Lily',
  'latin_name_enter_the_latin_name': 'Zantedeschia',
  'type_select_whether_the_plant_i': 'Shrub',
  'height_meters_enter_the_height': 0.4},
 8: {'common_name_enter_the_common_na': 'Rubber plant',
  'latin_name_enter_the_latin_name': 'Ficus elastica',
  'type_select_whether_the_plant_i': 'Tree',
  'height_meters_enter_the_height': 1.6}}